In [1]:
!pip uninstall -y surya-ocr surya || true
!pip install -q git+https://github.com/VikParuchuri/surya.git
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

In [ ]:
!pip uninstall -y transformers
!pip install -q transformers==4.57.3

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00


In [ ]:
import transformers
print(transformers.__version__)

4.57.3


In [ ]:
import os
import io
import json
import shutil
import uuid
import threading
import time
from pathlib import Path

import nest_asyncio
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
import requests

nest_asyncio.apply()

BASE_DIR = Path("/content/financial_ocr_server")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"

for d in [BASE_DIR, INPUT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Surya README says device can be overridden with TORCH_DEVICE
os.environ["TORCH_DEVICE"] = "cuda"
os.environ["RECOGNITION_BATCH_SIZE"] = "64"

print("Setup complete")
print("Base dir:", BASE_DIR)
print("TORCH_DEVICE:", os.environ.get("TORCH_DEVICE"))

Setup complete
Base dir: /content/financial_ocr_server
TORCH_DEVICE: cuda


In [ ]:
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

print("Surya predictors loaded successfully")

Surya predictors loaded successfully


In [ ]:
def clear_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def save_upload_file(upload: UploadFile, save_path: Path):
    with open(save_path, "wb") as f:
        f.write(upload.file.read())


def pil_from_path(path: Path) -> Image.Image:
    img = Image.open(path)
    # Ensure a safe RGB image for OCR
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    elif img.mode == "L":
        img = img.convert("RGB")
    return img


def normalize_polygon(poly):
    """
    Convert polygon-like objects into plain python lists if possible.
    """
    if poly is None:
        return None
    try:
        if hasattr(poly, "tolist"):
            return poly.tolist()
        return list(poly)
    except Exception:
        return poly


def normalize_bbox(bbox):
    if bbox is None:
        return None
    try:
        if hasattr(bbox, "tolist"):
            return bbox.tolist()
        return list(bbox)
    except Exception:
        return bbox


def prediction_to_text_lines(prediction):
    """
    Convert Surya prediction object into JSON-safe text lines.
    The README states OCR results include text, confidence, polygon, bbox.
    """
    text_lines = []

    lines = getattr(prediction, "text_lines", None) or []
    for line in lines:
        text_lines.append({
            "text": getattr(line, "text", "") or "",
            "confidence": float(getattr(line, "confidence", 0.0) or 0.0),
            "polygon": normalize_polygon(getattr(line, "polygon", None)),
            "bbox": normalize_bbox(getattr(line, "bbox", None)),
        })

    full_text = "\n".join(
        tl["text"].strip() for tl in text_lines if tl["text"].strip()
    ).strip()

    return {
        "page": 0,
        "text": full_text,
        "text_lines": text_lines,
    }


def run_surya_ocr_python(input_path: Path):
    """
    Run OCR through Surya Python API instead of CLI.
    """
    image = pil_from_path(input_path)

    predictions = recognition_predictor(
        [image],
        det_predictor=detection_predictor
    )

    if not predictions:
        raise ValueError("No predictions returned by Surya.")

    pred = predictions[0]
    page_data = prediction_to_text_lines(pred)

    raw_like = {
        input_path.stem: [
            {
                "page": 0,
                "text_lines": page_data["text_lines"]
            }
        ]
    }

    return {
        "raw_results": raw_like,
        "pages": [page_data],
        "text": page_data["text"],
    }

In [ ]:
app = FastAPI(title="Financial OCR API - Surya Python API")

@app.get("/")
def root():
    return {
        "message": "Financial OCR API is running",
        "engine": "Surya OCR (Python API)",
        "endpoint": "/ocr"
    }

@app.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
@app.post("/ocr")
async def ocr_endpoint(
    orig: UploadFile = File(None),
    p_img: UploadFile = File(None),
    m_img: UploadFile = File(None),
):
    if not any([orig, p_img, m_img]):
        return JSONResponse(
            status_code=400,
            content={
                "success": False,
                "error": "No files uploaded. Expected orig, p_img, or m_img"
            }
        )

    job_id = str(uuid.uuid4())[:8]
    job_input_dir = INPUT_DIR / job_id
    job_output_dir = OUTPUT_DIR / job_id
    job_input_dir.mkdir(parents=True, exist_ok=True)
    job_output_dir.mkdir(parents=True, exist_ok=True)

    uploads = {
        "orig": orig,
        "P": p_img,
        "M": m_img,
    }

    versions = {}
    failures = {}

    for version_name, upload in uploads.items():
        if upload is None:
            continue

        try:
            original_filename = upload.filename or f"{version_name}.png"
            input_file_path = job_input_dir / f"{version_name}__{original_filename}"

            save_upload_file(upload, input_file_path)

            result = run_surya_ocr_python(input_file_path)

            versions[version_name] = {
                "text": result["text"],
                "pages": result["pages"],
                "raw": result["raw_results"],
            }

        except Exception as e:
            failures[version_name] = str(e)

    return JSONResponse({
        "success": True,
        "job_id": job_id,
        "versions": versions,
        "failures": failures,
    })

In [ ]:
def run_api():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

time.sleep(5)
print("Server thread started")

INFO:     Started server process [1779]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server thread started


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:8000/health", timeout=10)
    print("Local health check:", r.status_code, r.text)
except Exception as e:
    print("Local health check failed:", e)

INFO:     127.0.0.1:56106 - "GET /health HTTP/1.1" 200 OK
Local health check: 200 {"status":"ok"}


In [ ]:
NGROK_AUTH_TOKEN = "3Aviwj4pxaGQj0LD4BmDb9Sgtlh_2ZfPm9fZNE92RiH6graF5"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(8000).public_url
print("Public URL:", public_url)
print("OCR endpoint:", public_url + "/ocr")
print("Health endpoint:", public_url + "/health")

Public URL: https://catechizable-uncongruously-armani.ngrok-free.dev
OCR endpoint: https://catechizable-uncongruously-armani.ngrok-free.dev/ocr
Health endpoint: https://catechizable-uncongruously-armani.ngrok-free.dev/health


In [ ]:
r = requests.get(public_url + "/health", timeout=20)
print("Ngrok health check:", r.status_code, r.text)

Ngrok health check: 404 <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />


In [ ]:
import threading
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


INFO:     Started server process [1779]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
